# VAE - ArtBench

https://www.geeksforgeeks.org/machine-learning/variational-autoencoders/

## Setup

In [3]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

from artbench_dataset import ArtBenchDataset, DATASET_PATH, download_artbench
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 32
BATCH_SIZE = 128

## Dataset

In [ ]:
from pathlib import Path

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

dataset = ArtBenchDataset(transform=transform)

subset_size = max(1, int(0.2 * len(dataset)))
subset, _ = random_split(dataset, [subset_size, len(dataset) - subset_size])

subset_size = int(0.2 * len(dataset))
subset, _ = random_split(dataset, [subset_size, len(dataset)-subset_size])

loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

A iniciar download do ArtBench...
Dataset URL: https://www.kaggle.com/datasets/alexanderliao/artbench10
artbench10.zip: Skipping, found more recently modified local copy (use --force to force download)
A extrair dataset...
Dataset pronto em: data/artbench-10-python e data/artbench-10-binary


FileNotFoundError: [WinError 3] O sistema não conseguiu localizar o caminho especificado: 'data\\artbench-10-python\\artbench-10-batches-py\\train'

## VAE Model

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,128,4,2,1), nn.ReLU()
        )
        self.fc_mu = nn.Linear(128*4*4,128)
        self.fc_logvar = nn.Linear(128*4*4,128)
        self.fc_dec = nn.Linear(128,128*4*4)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1), nn.Tanh()
        )

    def forward(self,x):
        h=self.enc(x).view(x.size(0),-1)
        mu,logvar=self.fc_mu(h),self.fc_logvar(h)
        z=mu+torch.randn_like(mu)*torch.exp(0.5*logvar)
        h=self.fc_dec(z).view(z.size(0),128,4,4)
        return self.dec(h),mu,logvar

## Training

In [ ]:
model=VAE().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=1e-3)

for epoch in range(5):
    total=0
    for x,_ in loader:
        x=x.to(DEVICE)
        recon,mu,logvar=model(x)
        loss=F.mse_loss(recon,x)-0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp())
        opt.zero_grad(); loss.backward(); opt.step()
        total+=loss.item()
    print(epoch,total/len(loader))